In [1]:
!pip install -q transformers datasets accelerate evaluate rouge-score
!pip install -q pymupdf pdfplumber Pillow

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import fitz 
from PIL import Image
import io
from datasets import load_dataset, DatasetDict
from transformers import (
    NougatProcessor, 
    VisionEncoderDecoderModel, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments,
    default_data_collator
)
import evaluate
device = "cuda" if torch.cuda.is_available() else "cpu"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 70.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 113.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.0 MB/s eta 0:00:00:00:01


In [2]:
!apt-get update -y
!apt-get install tesseract-ocr -y
!pip install -q pytesseract

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,646 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,0

In [3]:
import os
from datasets import load_dataset, DatasetDict
dataset = load_dataset("marcodsn/arxiv-markdown", split="train[:500]")
def check_local_pdf(example):
    pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-big/arxiv_dataset_pdfs/", f"{example['arxiv_id']}.pdf")
    return os.path.exists(pdf_path)
dataset = dataset.filter(check_local_pdf)

total_found = len(dataset)
if total_found == 0:
    raise FileNotFoundError("404 Not Found.")

test_size = 0.15 
valid_size = 0.15
train_test_split = dataset.train_test_split(test_size=test_size, seed=42)
test_dataset = train_test_split['test']
train_valid_split = train_test_split['train'].train_test_split(test_size=valid_size, seed=42)
train_dataset = train_valid_split['train']
valid_dataset = train_valid_split['test']
final_datasets = DatasetDict({
    'train': train_dataset,
    'valid': valid_dataset,
    'test': test_dataset
})
final_datasets.set_format("torch")

print(f"Train: {len(final_datasets['train'])} mẫu")
print(f"Valid: {len(final_datasets['valid'])} mẫu")
print(f"Test: {len(final_datasets['test'])} mẫu")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/96.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 361 mẫu
Valid: 64 mẫu
Test: 75 mẫu


In [4]:
import os
import io
from PIL import Image
import fitz 
from transformers import NougatProcessor
processor = NougatProcessor.from_pretrained("facebook/nougat-small", use_fast=False)

available_columns = final_datasets['train'].column_names
content_col = 'markdown' if 'markdown' in available_columns else 'content'

def preprocess_function(examples):
    pixel_values, labels = [], []
    batch_arxiv_ids = examples['arxiv_id']
    batch_markdowns = examples[content_col]
    
    for arxiv_id, markdown_text in zip(batch_arxiv_ids, batch_markdowns):
        pdf_path = os.path.join("/kaggle/input/datasets/danghiennguyen/arxiv-pdf-big/arxiv_dataset_pdfs/", f"{arxiv_id}.pdf")
        if not os.path.exists(pdf_path):
            print(f"Skip {arxiv_id}")
            continue
            
        try:
            doc = fitz.open(pdf_path)
            page = doc.load_page(0) 
            pix = page.get_pixmap(dpi=150)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            doc.close()
            processed_img = processor(img, return_tensors="pt").pixel_values.squeeze()
            pixel_values.append(processed_img)
            tokenized_text = processor.tokenizer(
                str(markdown_text)[:1024], padding="max_length", 
                max_length=512, truncation=True, return_tensors="pt"
            ).input_ids.squeeze()
            tokenized_text[tokenized_text == processor.tokenizer.pad_token_id] = -100
            labels.append(tokenized_text)
            
        except Exception as e:
            print(f"Skip{arxiv_id}: {e}")
            continue
            
    return {"pixel_values": pixel_values, "labels": labels}
train_dataset = final_datasets['train'].map(preprocess_function, batched=True, remove_columns=available_columns)
valid_dataset = final_datasets['valid'].map(preprocess_function, batched=True, remove_columns=available_columns)
test_dataset = final_datasets['test'].map(preprocess_function, batched=True, remove_columns=available_columns)

preprocessor_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Map:   0%|          | 0/361 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

In [5]:
rouge = evaluate.load("rouge")
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return result

In [6]:
num_tasks = 5
train_chunk_size = len(train_dataset) // num_tasks
test_chunk_size = len(test_dataset) // num_tasks

task_train_datasets = []
task_test_datasets = []

for i in range(num_tasks):
    train_start = i * train_chunk_size
    train_end = len(train_dataset) if i == num_tasks - 1 else (i + 1) * train_chunk_size
    task_train_datasets.append(train_dataset.select(range(train_start, train_end)))
    test_start = i * test_chunk_size
    test_end = len(test_dataset) if i == num_tasks - 1 else (i + 1) * test_chunk_size
    task_test_datasets.append(test_dataset.select(range(test_start, test_end)))

print(f"Split {num_tasks} Tasks.")
for i in range(num_tasks):
    print(f"Task {i+1} - Train: {len(task_train_datasets[i])} mẫu, Test: {len(task_test_datasets[i])} mẫu")

Split 5 Tasks.
Task 1 - Train: 72 mẫu, Test: 15 mẫu
Task 2 - Train: 72 mẫu, Test: 15 mẫu
Task 3 - Train: 72 mẫu, Test: 15 mẫu
Task 4 - Train: 72 mẫu, Test: 15 mẫu
Task 5 - Train: 73 mẫu, Test: 15 mẫu


In [7]:
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, VisionEncoderDecoderModel, default_data_collator
accuracy_matrix = np.zeros((num_tasks, num_tasks))
average_accuracies = []
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")
start_token = processor.tokenizer.bos_token_id
if start_token is None:
    start_token = processor.tokenizer.convert_tokens_to_ids("<s>")
model.config.decoder_start_token_id = start_token
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder.decoder_start_token_id = start_token
model.config.decoder.pad_token_id = processor.tokenizer.pad_token_id
for t in range(num_tasks):
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./ocr_cl_task_{t+1}",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=3e-5,
        num_train_epochs=2,
        predict_with_generate=True,
        report_to="none",
        save_strategy="no", 
        logging_steps=10
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=task_train_datasets[t],
        processing_class=processor.tokenizer,
        data_collator=default_data_collator,   
        compute_metrics=compute_metrics,
    )

      
    trainer.train()
    print(f"\nTask {t+1}:")
    model.config.use_cache = True
    if hasattr(model.config, 'decoder'):
        model.config.decoder.use_cache = True

    for tau in range(t + 1):
        print(f"  ->Test data on Task {tau+1}...")
        test_results = trainer.predict(task_test_datasets[tau])
        acc = test_results.metrics['test_rougeL'] 
        accuracy_matrix[t, tau] = acc
        print(f"ROUGE-L (Task {tau+1}): {acc:.4f}")
    row_accuracies = accuracy_matrix[t, :t+1]
    aa_t = np.mean(row_accuracies)
    average_accuracies.append(aa_t)
    print(f"\nAVERAGE ACCURACY (AA_{t+1}): {aa_t:.4f}\n")

trainer.save_model("./best_cl_ocr_model")
processor.save_pretrained("./best_cl_ocr_model")

print("CONTINUAL LEARNING (ROUGE-L) RESULT METRICS:")
print(np.round(accuracy_matrix, 4))
print("\nAverage Accuracy (AA):")
for i, aa in enumerate(average_accuracies):
    print(f"- Sau Task {i+1}: {aa:.4f}")
print("*"*50)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.


Step,Training Loss
10,0.805040
20,0.547254
30,1.237781
40,0.586311
50,0.735958
60,0.413696
70,0.675344


...

ROUGE-L (Task 3): 0.1112
  ->Test data on Task 4...


ROUGE-L (Task 4): 0.1185
  ->Test data on Task 5...


ROUGE-L (Task 5): 0.1258

AVERAGE ACCURACY (AA_5): 0.1175



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CONTINUAL LEARNING (ROUGE-L) RESULT METRICS:
[[0.1013 0.     0.     0.     0.    ]
 [0.1067 0.0983 0.     0.     0.    ]
 [0.1081 0.1103 0.1045 0.     0.    ]
 [0.1055 0.1194 0.1114 0.1267 0.    ]
 [0.1063 0.1254 0.1112 0.1185 0.1258]]

Average Accuracy (AA):
- Sau Task 1: 0.1013
- Sau Task 2: 0.1025
- Sau Task 3: 0.1076
- Sau Task 4: 0.1158
- Sau Task 5: 0.1175
**************************************************
